# scicp — Fine-tune Scripture Embedding Model (Kaggle)

Fine-tunes a chosen base model on scripture training pairs using **MultipleNegativesRankingLoss**.

Supported base models (configure in cell 2):

| Key | Model | Dims | Params | Notes |
|-----|-------|------|--------|-------|
| `minilm-l6` | all-MiniLM-L6-v2 | 384 | 22M | Fastest inference, smallest footprint |
| `minilm-l12` | all-MiniLM-L12-v2 | 384 | 33M | ~2× slower than L6, richer layers |
| `mpnet` | all-mpnet-base-v2 | 768 | 109M | Best quality/speed balance |
| `distilroberta` | all-distilroberta-v1 | 768 | 82M | Not recommended — see notes |
| `bge-base` | BAAI/bge-base-en-v1.5 | 768 | 109M | Top MTEB score, mean-pool override |
| `bge-large` | BAAI/bge-large-en-v1.5 | 1024 | 335M | Highest quality, slow on CPU |

**Setup:**
1. Create a Kaggle Dataset called `scicp-training` and upload `training-pairs.json`
2. In notebook settings: choose a GPU accelerator (`T4`, `P100`, or `A100`)
3. Add the dataset: **+ Add Data → Your Datasets → scicp-training**
4. Set `MODEL_CHOICE` in cell 2, then **Run All**
5. Download `scripture-bge.zip` from the Output tab

**Important GPU note:**
- Standard Kaggle notebook execution is **not** a proper multi-GPU DDP launch.
- If Kaggle exposes multiple GPUs, this notebook intentionally trains on a single GPU unless you explicitly launch distributed training outside the notebook.
- This avoids the slower `DataParallel` fallback and reduces `bge-large` OOM risk on T4s.

**Training profiles:**
- `fast`: faster iteration for `bge-large` by using 1 epoch, shorter sequences, a capped training subset, and no per-epoch validation/checkpoint overhead.
- `full`: slower, higher-confidence training for the final model before rebaking production search artifacts.

**Estimated training time:**
- `minilm-l6` / `minilm-l12`: ~15–20 min
- `mpnet` / `distilroberta` / `bge-base`: ~35–50 min
- `bge-large` in `fast` profile on T4 single-GPU mode: ~2–5 hours
- `bge-large` in `full` profile on T4 single-GPU mode: often too slow for comfortable iteration; prefer P100 or A100

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
%pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. CONFIGURATION — edit this cell only ───────────────────────────────────
#
# Choose your base model:
#   'minilm-l6'     → all-MiniLM-L6-v2        (384-dim, fastest)
#   'minilm-l12'    → all-MiniLM-L12-v2       (384-dim, richer than L6)
#   'mpnet'         → all-mpnet-base-v2        (768-dim, best quality/speed tradeoff)
#   'distilroberta' → all-distilroberta-v1     (768-dim, not recommended)
#   'bge-base'      → BAAI/bge-base-en-v1.5   (768-dim, top MTEB, mean-pool override)
#   'bge-large'     → BAAI/bge-large-en-v1.5  (1024-dim, highest quality, slow)
#
MODEL_CHOICE = 'bge-base'

# Keep the exported artifact name stable so local deployment scripts and
# wrapper defaults continue to use scripture-bge.zip.
OUTPUT_NAME = 'scripture-bge'

# Training profile:
#   'fast' → faster iteration, reduced runtime for bge-large
#   'full' → slower final-quality run before a production rebake
TRAINING_PROFILE = 'full'

# Learning rate — use 1e-5 for bge-large, 2e-5 for the smaller models here
LEARNING_RATE = 2e-5
# Auto-tune options: set to True to let the notebook pick batch sizes based on VRAM.
AUTO_TUNE = True
# Force BF16 even on smaller GPUs (use with care). If False, BF16 will be suggested on large VRAM GPUs.
FORCE_BF16 = False
print(f"Auto-tune      : {AUTO_TUNE}")
print(f"Force bf16     : {FORCE_BF16}")

# ─────────────────────────────────────────────────────────────────────────────
# Model registry — do not edit unless adding a new model
MODEL_REGISTRY = {
    'minilm-l6':     {'hf_id': 'sentence-transformers/all-MiniLM-L6-v2',     'dims': 384,  'bge': False},
    'minilm-l12':    {'hf_id': 'sentence-transformers/all-MiniLM-L12-v2',    'dims': 384,  'bge': False},
    'mpnet':         {'hf_id': 'sentence-transformers/all-mpnet-base-v2',     'dims': 768,  'bge': False},
    'distilroberta': {'hf_id': 'sentence-transformers/all-distilroberta-v1',  'dims': 768,  'bge': False},
    'bge-base':      {'hf_id': 'BAAI/bge-base-en-v1.5',                       'dims': 768,  'bge': True},
    'bge-large':     {'hf_id': 'BAAI/bge-large-en-v1.5',                      'dims': 1024, 'bge': True},
}

PROFILE_DEFAULTS = {
    'fast': {
        'epochs': 1,
        'max_seq_length': 192,
        'max_train_pairs': 200_000,
        'validation_fraction': 0.02,
        'run_validation': False,
    },
    'full': {
        'epochs': 2,
        'max_seq_length': 256,
        'max_train_pairs': None,
        'validation_fraction': 0.03,
        'run_validation': True,
    },
}

assert MODEL_CHOICE in MODEL_REGISTRY, (
    f"Unknown MODEL_CHOICE '{MODEL_CHOICE}'. Valid options: {list(MODEL_REGISTRY.keys())}"
)
assert TRAINING_PROFILE in PROFILE_DEFAULTS, (
    f"Unknown TRAINING_PROFILE '{TRAINING_PROFILE}'. Valid options: {list(PROFILE_DEFAULTS.keys())}"
)

cfg = MODEL_REGISTRY[MODEL_CHOICE]
profile = PROFILE_DEFAULTS[TRAINING_PROFILE]

# Default bge-large to the faster profile because Kaggle single-GPU notebook
# execution is otherwise too slow for normal iteration.
if MODEL_CHOICE == 'bge-large' and TRAINING_PROFILE == 'full':
    print("Using full bge-large profile — expect substantially longer runtime.")

EPOCHS = profile['epochs']
MAX_SEQ_LENGTH = profile['max_seq_length']
MAX_TRAIN_PAIRS = profile['max_train_pairs']
VALIDATION_FRACTION = profile['validation_fraction']
RUN_VALIDATION = profile['run_validation']

print(f"Selected model : {cfg['hf_id']}")
print(f"Embedding dims : {cfg['dims']}")
print(f"BGE pooling fix: {cfg['bge']}")
print(f"Output name    : {OUTPUT_NAME}")
print(f"Profile        : {TRAINING_PROFILE}")
print(f"Epochs         : {EPOCHS}  |  LR: {LEARNING_RATE}")
print(f"Max seq length : {MAX_SEQ_LENGTH}")
print(f"Max train pairs: {MAX_TRAIN_PAIRS}")
print(f"Run validation : {RUN_VALIDATION}")

In [ ]:
# ── 3. Detect GPU + choose a safe training profile ───────────────────────────
import os

80
40
40
40
22
22
22

40
22
22

RAW_VISIBLE_DEVICES = os.environ.get('CUDA_VISIBLE_DEVICES', '').strip()

# Kaggle notebook sessions may expose multiple GPUs even when the notebook is
# not launched under true DDP. In that case, mask to GPU 0 before importing
# torch so Trainer does not fall back to slow DataParallel.
if not IS_DISTRIBUTED and RAW_VISIBLE_DEVICES and ',' in RAW_VISIBLE_DEVICES:
    os.environ['CUDA_VISIBLE_DEVICES'] = RAW_VISIBLE_DEVICES.split(',')[0].strip()
    print(f"Notebook multi-GPU detected; masking to CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} for single-GPU training.")
import torch

VISIBLE_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_SINGLE_GPU_NOTEBOOK_MODE = torch.cuda.is_available() and VISIBLE_GPUS > 1 and not IS_DISTRIBUTED

print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs   :", VISIBLE_GPUS)
print("Distributed run:", IS_DISTRIBUTED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

    if USE_SINGLE_GPU_NOTEBOOK_MODE:
        # Fallback guard if the environment still exposes more than one GPU.
        torch.cuda.set_device(0)
        TRAIN_GPU_COUNT = 1
        print("Trainer fallback guard: forcing single-GPU training to avoid DataParallel.")
    else:
        TRAIN_GPU_COUNT = max(1, VISIBLE_GPUS)

    print(f"Primary GPU    : {gpu_name}")
    print(f"VRAM / GPU     : {vram_gb} GB")

    # Batch size table: {model_key: {vram_tier: (micro_batch, grad_accum)}}
    BATCH_TABLE = {
        'minilm-l6':     {40: (512, 1), 22: (256, 1), 0: (128, 2)},
        'minilm-l12':    {40: (512, 1), 22: (256, 1), 0: (128, 2)},
        'mpnet':         {40: (256, 1), 22: (128, 1), 0: (64,  2)},
        'distilroberta': {40: (256, 1), 22: (128, 1), 0: (64,  2)},
        'bge-base':      {40: (192, 1), 22: (96,  1), 0: (48,  2)},
        'bge-large':     {40: (64,  2), 22: (24,  4), 0: (8,  16)},
    }

    tiers = BATCH_TABLE[MODEL_CHOICE]
    if vram_gb >= 40:
        MICRO_BATCH, GRAD_ACCUM = tiers[40]
    elif vram_gb >= 22:
        MICRO_BATCH, GRAD_ACCUM = tiers[22]
    else:
        MICRO_BATCH, GRAD_ACCUM = tiers[0]

else:
    print("No GPU detected — training will be extremely slow")
    TRAIN_GPU_COUNT = 0
    MICRO_BATCH, GRAD_ACCUM = 8, 32
    vram_gb = 0

USE_GRADIENT_CHECKPOINTING = MODEL_CHOICE == 'bge-large'
effective_batch = MICRO_BATCH * GRAD_ACCUM

print(f"Training GPUs   : {TRAIN_GPU_COUNT}")
print(f"Micro batch     : {MICRO_BATCH}")
print(f"Grad accum steps: {GRAD_ACCUM}")
print(f"Effective batch : {effective_batch}  (= in-batch negatives + 1 per sample)")
print(f"Grad checkpoint : {USE_GRADIENT_CHECKPOINTING}")

In [ ]:
# ── 4. Locate training data ──────────────────────────────────────────────────
import os, glob, time

print("/kaggle/input contents:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(" ", os.path.join(root, f))

# Recursive search handles both flat and nested Kaggle dataset layouts
candidates = glob.glob("/kaggle/input/**/training-pairs.json", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "training-pairs.json not found under /kaggle/input/.\n"
        "Make sure you added the dataset via + Add Data → Your Datasets."
    )
LOCAL_PATH = candidates[0]
print(f"\nUsing : {LOCAL_PATH}")
print(f"Size  : {os.path.getsize(LOCAL_PATH) // 1024 // 1024} MB")

In [ ]:
# ── 5. Load + prepare dataset ────────────────────────────────────────────────
import json, random
from datasets import Dataset

t0 = time.time()
with open(LOCAL_PATH) as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs):,} pairs in {time.time() - t0:.1f}s")

random.seed(42)
random.shuffle(pairs)

if MAX_TRAIN_PAIRS is not None and len(pairs) > MAX_TRAIN_PAIRS:
    print(f"Capping dataset to {MAX_TRAIN_PAIRS:,} pairs for {TRAINING_PROFILE} profile")
    pairs = pairs[:MAX_TRAIN_PAIRS]

split = int(len(pairs) * (1 - VALIDATION_FRACTION))
split = max(1, min(split, len(pairs) - 1))
train_pairs = pairs[:split]
val_pairs = pairs[split:]

# Include hard_negative column when the dataset has it (mine-hard-negatives.js output).
# sentence-transformers uses it as an explicit hard negative in addition to in-batch negatives.
_has_hard_neg = bool(train_pairs) and 'hard_negative' in train_pairs[0]
print(f"Hard negatives: {'yes' if _has_hard_neg else 'no (not present in dataset)'}")

_td = {'anchor': [p['anchor'] for p in train_pairs], 'positive': [p['positive'] for p in train_pairs]}
if _has_hard_neg:
    _td['negative'] = [p['hard_negative'] for p in train_pairs]
train_ds = Dataset.from_dict(_td)

if RUN_VALIDATION and val_pairs:
    _vd = {'anchor': [p['anchor'] for p in val_pairs], 'positive': [p['positive'] for p in val_pairs]}
    if _has_hard_neg:
        _vd['negative'] = [p['hard_negative'] for p in val_pairs]
    val_ds = Dataset.from_dict(_vd)
else:
    val_ds = None

del pairs  # free RAM (~200–400 MB depending on dataset size)

print(f"Train : {len(train_ds):,}")
print(f"Val   : {len(val_ds):,}" if val_ds is not None else "Val   : skipped in fast profile")
print(f"Sample: {train_ds[0]}")

In [ ]:
# ── 6. Build model ───────────────────────────────────────────────────────────
#
# BGE models use CLS-token pooling by default, which is suboptimal for
# MultipleNegativesRankingLoss fine-tuning. Gradients only flow through a
# single aggregation token rather than the full sequence. We override to
# mean pooling, which uses all token representations and trains more stably.
#
# For all other models (MiniLM, mpnet, distilroberta), mean pooling is
# already the default — we load them normally.

from sentence_transformers import SentenceTransformer, models, losses

if cfg['bge']:
    print(f"BGE model detected — overriding CLS pooling → mean pooling")
    word_model = models.Transformer(cfg['hf_id'])
    pooling = models.Pooling(
        word_model.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
    )
    model = SentenceTransformer(modules=[word_model, pooling])
else:
    model = SentenceTransformer(cfg['hf_id'])

model.max_seq_length = MAX_SEQ_LENGTH
print(f"Max sequence len: {model.max_seq_length}")

# Verify the embedding dimension matches expectations
test_emb = model.encode("And it came to pass")
print(f"Model loaded    : {cfg['hf_id']}")
print(f"Embedding shape : {test_emb.shape}  (expected: {cfg['dims']})")
assert test_emb.shape[0] == cfg['dims'], "Dimension mismatch — check MODEL_REGISTRY entry"

loss = losses.MultipleNegativesRankingLoss(model)
print("Loss            : MultipleNegativesRankingLoss")

In [ ]:
# ── 7. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

OUT_DIR = f"/kaggle/working/{OUTPUT_NAME}"

total_steps = (len(train_ds) // effective_batch) * EPOCHS
warmup_steps = max(1, total_steps // 20) if total_steps else 0  # 5% warmup

print(f"Output dir      : {OUT_DIR}")
print(f"Epochs          : {EPOCHS}")
print(f"Steps/epoch     : {len(train_ds) // effective_batch:,}")
print(f"Total steps     : {total_steps:,}")
print(f"Warmup steps    : {warmup_steps}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"In-batch negatives per sample: {effective_batch - 1}")
print(f"Validation mode : {RUN_VALIDATION}")

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    per_device_eval_batch_size=max(1, MICRO_BATCH * 2),
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch" if RUN_VALIDATION else "no",
    save_strategy="epoch" if RUN_VALIDATION else "no",
    save_total_limit=2 if RUN_VALIDATION else 1,
    load_best_model_at_end=RUN_VALIDATION,
    logging_steps=50,
    fp16=(torch.cuda.is_available() and not USE_BF16),
    bf16=USE_BF16,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available(),
    metric_for_best_model="eval_loss" if RUN_VALIDATION else None,
    greater_is_better=False if RUN_VALIDATION else None,
    report_to="none",
)

if torch.cuda.is_available() and USE_SINGLE_GPU_NOTEBOOK_MODE:
    # Prevent Trainer from wrapping the model in DataParallel inside a standard
    # notebook session. If you want true multi-GPU speedup, launch the notebook
    # code with torchrun/accelerate so it uses DDP instead.
    args._n_gpu = 1
    print("Notebook mode: forcing Trainer single-GPU path; DDP is only used in explicit distributed launches.")

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds if RUN_VALIDATION else None,
    loss=loss,
    )

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
total_pairs_seen = len(train_ds) * EPOCHS
print(f"\nTraining complete")
print(f"   Time          : {elapsed:.1f} min")
print(f"   Pairs seen    : {total_pairs_seen:,}")
print(f"   Throughput    : {total_pairs_seen / elapsed:,.0f} pairs/min")

In [ ]:
# ── 8. Smoke-test the fine-tuned model ──────────────────────────────────────
#
# A quick sanity check before saving. We verify that theologically related
# verses score higher than unrelated ones. If scores look inverted or flat,
# something went wrong in training.

from sentence_transformers.util import cos_sim

probe_pairs = [
    # Should be HIGH similarity
    ("For God so loved the world",         "God's love for humanity"),
    ("I am the bread of life",              "manna from heaven in the wilderness"),
    ("faith without works is dead",         "true belief requires action"),
    # Should be LOW similarity
    ("For God so loved the world",          "the dimensions of Noah's ark"),
    ("I am the resurrection and the life",  "a census of the tribes of Israel"),
]

print("Smoke test — cosine similarities after fine-tuning:\n")
for a, b in probe_pairs:
    emb_a = model.encode(a, convert_to_tensor=True)
    emb_b = model.encode(b, convert_to_tensor=True)
    score = cos_sim(emb_a, emb_b).item()
    bar   = '█' * int(score * 30)
    print(f"  {score:.3f} {bar}")
    print(f"         A: {a}")
    print(f"         B: {b}")
    print()

In [ ]:
# ── 9. Save model + create download zip ─────────────────────────────────────
import shutil

OUT_DIR  = f"/kaggle/working/{OUTPUT_NAME}"
ZIP_BASE = f"/kaggle/working/{OUTPUT_NAME}"
ZIP_PATH = f"{ZIP_BASE}.zip"

# Save the best checkpoint (loaded automatically by load_best_model_at_end)
model.save(OUT_DIR)

model_files = [f for f in os.listdir(OUT_DIR) if not f.startswith('checkpoint')]
print("Model files saved:")
for f in sorted(model_files):
    size = os.path.getsize(os.path.join(OUT_DIR, f)) // 1024
    print(f"  {f:40s}  {size:>6} KB")

# Zip for easy download from the Kaggle Output tab
shutil.make_archive(ZIP_BASE, 'zip', OUT_DIR)
zip_mb = os.path.getsize(ZIP_PATH) // 1024 // 1024
print(f"\n✅ {OUTPUT_NAME}.zip ({zip_mb} MB) is ready in the Output tab")
print(f"   Extract to: resources/models/{OUTPUT_NAME}/")

## After training — local deployment

Download `scripture-bge.zip` from the **Output** tab (right side panel).

On your local machine, use the wrapper that matches the current production search pipeline:

```bash
# Rebuild all embedding-dependent artifacts from the new model zip
scripts/post-train-rebuild.sh /path/to/scripture-bge.zip
```

This wrapper currently performs the full post-train flow in the correct order:

1. Install the fine-tuned model into `resources/models/scripture-bge/`
2. Re-encode all verses with `scripts/rebake-embeddings.py`
3. Rebuild the concept index
4. Rebuild SVD features
5. Rebuild the kNN graph
6. Rebuild spectral embeddings
7. Rebuild clusters and cluster labels
8. Rebuild entity centroids
9. Rebuild the HNSW index
10. Rebuild the packaged search graph bundle

### Important

- Do **not** run whitening. `prebake-whitening.js` is deprecated and intentionally excluded because whitening was previously inverting semantic similarity rankings.
- If you switch embedding families or dimensions, make sure the local rebake scripts and runtime storage still support the selected model before deploying it.
- After the wrapper finishes, restart the backend with `npm run dev --workspace=backend`.